In [10]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using:", device)

# Smaller settings because this is running on a CPU Chromebook
batch_size = 16
block_size = 32
max_iters = 1000
eval_interval = 100
learning_rate = 3e-4
eval_iters = 50

n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.2

Using: cpu


In [11]:
with open('The_Odyssey.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Dataset length:", len(text))
print("Vocabulary size:", vocab_size)

Dataset length: 21512
Vocabulary size: 75


In [12]:
string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

print(data[:100])

tensor([39, 52, 49,  1, 35, 48, 69, 63, 63, 49, 69,  0,  0, 46, 69,  1, 28, 59,
        57, 49, 62,  0,  0,  0, 23, 59, 58, 64, 49, 58, 64, 63,  0,  0,  1, 39,
        28, 25,  1, 35, 24, 43, 38, 38, 25, 43,  0,  1, 22, 35, 35, 31,  1, 29,
         7,  0,  0, 39, 28, 25,  1, 35, 24, 43, 38, 38, 25, 43,  0,  0,  0, 22,
        35, 35, 31,  1, 29,  0,  0,  0, 39, 28, 25,  1, 27, 35, 24, 38,  1, 29,
        34,  1, 23, 35, 40, 34, 23, 29, 32, 71])


In [13]:
n = int(0.8 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Training characters:", len(train_data))
print("Validation characters:", len(val_data))

Training characters: 17209
Validation characters: 4303


In [14]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    x, y = x.to(device), y.to(device)

    return x, y

In [15]:
@torch.no_grad()
def estimate_loss():

    out = {}

    model.eval()

    for split in ['train', 'val']:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [16]:
class Head(nn.Module):
    """One head of self-attention."""

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        # Attention scores
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5

        # Prevent tokens from looking into the future
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)

        out = wei @ v

        return out

In [17]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            head_size * num_heads,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        out = self.dropout(
            self.proj(out)
        )

        return out

In [18]:
class FeedForward(nn.Module):
    """A simple linear layer followed by a non-linearity."""

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                n_embd,
                4 * n_embd
            ),

            nn.ReLU(),

            nn.Linear(
                4 * n_embd,
                n_embd
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [19]:
class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        x = x + self.sa(self.ln1(x))

        x = x + self.ffwd(self.ln2(x))

        return x

In [20]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Positional embeddings
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head=n_head)
                for _ in range(n_layer)
            ]
        )

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Convert embeddings into vocabulary predictions
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )

        self.apply(self._init_weights)


    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                torch.nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )


    def forward(self, index, targets=None):

        B, T = index.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(index)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        # Combine token + position information
        x = tok_emb + pos_emb

        # Transformer
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Predictions
        logits = self.lm_head(x)

        if targets is None:

            loss = None

        else:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss


    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            # Keep only the most recent block_size tokens
            index_cond = index[:, -block_size:]

            logits, loss = self.forward(index_cond)

            # Look only at final token
            logits = logits[:, -1, :]

            # Convert logits into probabilities
            probs = F.softmax(
                logits,
                dim=-1
            )

            # Sample next token
            index_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add prediction to sequence
            index = torch.cat(
                (index, index_next),
                dim=1
            )

        return index

In [21]:
model = GPTLanguageModel().to(device)

num_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Model parameters: {num_parameters:,}"
)

Model parameters: 211,019


In [22]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated = model.generate(
    context,
    max_new_tokens=200
)

print(
    decode(
        generated[0].tolist()
    )
)


O—8 aC,.vjIRkCu4CgDoiJ7,vaGOmWgq.-K.
Mlo17?rwYmhdDD2swm31TvWAe4,chHkZT5sAFPjm)Jp8zw-LcBJb.FPnS:?CqeGvuUE7EZ4Rx3uy?MEUsZHNSH55kEoVv1-TRK5Ex.UOM9gmZtdEEL)E0—hpF1FJeOnO:Kl-mgtKqnE0qAr) IqcUi:to1)W;.tme””


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for iteration in range(max_iters):

    if iteration % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"step {iteration}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

print(
    "Final loss:",
    loss.item()
)

step 0: train loss 4.3022, val loss 4.3053
step 100: train loss 2.9348, val loss 2.9328
step 200: train loss 2.6074, val loss 2.6148


In [ ]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated = model.generate(
    context,
    max_new_tokens=500
)

generated_text = decode(
    generated[0].tolist()
)

print(generated_text)